# 💰 Price Comparisons Analysis

This notebook explores different aspects of flight pricing to help users find the best deals.

## Setup: Import Libraries and Load Data

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Load the dataset
df = pd.read_excel('airline_ticket_dataset.xlsx')
print(f"✅ Loaded {len(df):,} flight records")
df.head()

✅ Loaded 14,004 flight records


,Year,quarter,citymarketid_1,citymarketid_2,city1,city2,nsmiles,passengers,fare,carrier_lg,...,fare_lg,carrier_low,lf_ms,fare_low,TotalFaredPax_city1,TotalPerLFMkts_city1,TotalPerPrem_city1,TotalFaredPax_city2,TotalPerLFMkts_city2,TotalPerPrem_city2
0,2025,2,32467,31703,"Miami, FL (Metropolitan Area)","New York City, NY (Metropolitan Area)",1118,17955,208.52,B6,...,191.48,B6,0.2551,191.48,4.322090e+06,0.88590,-0.065800,2.978413e+06,0.821150,-0.032867
1,2025,2,32575,32457,"Los Angeles, CA (Metropolitan Area)","San Francisco, CA (Metropolitan Area)",372,17310,157.68,WN,...,169.03,AS,0.1193,140.59,2.822942e+06,0.86508,-0.038060,3.284783e+06,0.880833,-0.051467
2,2025,2,32575,31703,"Los Angeles, CA (Metropolitan Area)","New York City, NY (Metropolitan Area)",2510,13648,430.38,DL,...,526.21,B6,0.2272,365.63,2.822942e+06,0.86508,-0.038060,2.978413e+06,0.821150,-0.032867
3,2025,2,31703,31454,"New York City, NY (Metropolitan Area)","Orlando, FL",989,12627,186.50,B6,...,186.10,B6,0.3735,186.10,2.978413e+06,0.82115,-0.032867,8.664260e+06,0.979700,-0.115500
4,2025,2,30977,31703,"Chicago, IL","New York City, NY (Metropolitan Area)",773,11284,221.33,UA,...,238.62,AA,0.2426,217.36,5.402975e+06,0.77515,-0.007850,2.978413e+06,0.821150,-0.032867


## 1. Average Fare by Route (Top 20)

Which routes are most expensive?

In [2]:
# Create route column
df['route'] = df['city1'] + ' → ' + df['city2']

# Calculate average fare by route
route_fares = df.groupby('route').agg({
    'fare': 'mean',
    'passengers': 'sum',
    'nsmiles': 'mean'
}).reset_index()

# Get top 20 most expensive routes
top_expensive = route_fares.nlargest(20, 'fare')

fig = px.bar(top_expensive, 
             x='fare', 
             y='route',
             orientation='h',
             title='Top 20 Most Expensive Routes',
             labels={'fare': 'Average Fare ($)', 'route': 'Route'},
             color='fare',
             color_continuous_scale='Reds',
             hover_data={'passengers': ':,', 'nsmiles': ':.0f'})

fig.update_layout(height=600, yaxis={'categoryorder':'total ascending'})
fig.show()

## 2. Cheapest Routes (Top 20)

Budget-friendly flight options:

In [3]:
# Get top 20 cheapest routes
top_cheap = route_fares.nsmallest(20, 'fare')

fig = px.bar(top_cheap, 
             x='fare', 
             y='route',
             orientation='h',
             title='Top 20 Cheapest Routes',
             labels={'fare': 'Average Fare ($)', 'route': 'Route'},
             color='fare',
             color_continuous_scale='Greens',
             hover_data={'passengers': ':,', 'nsmiles': ':.0f'})

fig.update_layout(height=600, yaxis={'categoryorder':'total descending'})
fig.show()

## 3. Fare Distribution Overview

How are flight prices spread out?

In [4]:
fig = px.histogram(df, 
                   x='fare',
                   nbins=60,
                   title='Distribution of Flight Fares',
                   labels={'fare': 'Fare ($)', 'count': 'Number of Flights'},
                   color_discrete_sequence=['#636EFA'])

# Add average line
avg_fare = df['fare'].mean()
fig.add_vline(x=avg_fare, line_dash="dash", line_color="red",
              annotation_text=f"Average: ${avg_fare:.2f}",
              annotation_position="top right")

fig.update_layout(showlegend=False)
fig.show()

print(f"📊 Average fare: ${df['fare'].mean():.2f}")
print(f"📊 Median fare: ${df['fare'].median():.2f}")
print(f"📊 Cheapest: ${df['fare'].min():.2f}")
print(f"📊 Most expensive: ${df['fare'].max():.2f}")

📊 Average fare: $237.70
📊 Median fare: $227.64
📊 Cheapest: $76.77
📊 Most expensive: $676.89


## 4. Fare vs Distance Analysis

Does flying farther cost more?

In [ ]:
# Sample data for better visualization (plot every 10th point)
df_sample = df.sample(n=min(2000, len(df)), random_state=42)

fig = px.scatter(df_sample, 
                 x='nsmiles', 
                 y='fare',
                 color='passengers',
                 size='passengers',
                 hover_data=['route', 'carrier_lg'],
                 title='Fare vs Distance (with Passenger Volume)',
                 labels={'nsmiles': 'Distance (miles)', 'fare': 'Fare ($)', 'passengers': 'Passengers'},
                 opacity=0.6,
                 color_continuous_scale='Viridis')

fig.update_layout(height=600)
fig.show()

# Calculate price per mile
df['price_per_mile'] = df['fare'] / df['nsmiles']
print(f"\n💡 Average price per mile: ${df['price_per_mile'].mean():.3f}")

ModuleNotFoundError: No module named 'statsmodels'

## 5. Large Carrier vs Low-Cost Carrier Pricing

How much can you save with budget airlines?

In [ ]:
# Create comparison data
comparison_data = pd.DataFrame({
    'Carrier Type': ['Large Carrier', 'Low-Cost Carrier'],
    'Average Fare': [df['fare_lg'].mean(), df['fare_low'].mean()]
})

fig = px.bar(comparison_data, 
             x='Carrier Type', 
             y='Average Fare',
             title='Average Fare: Large vs Low-Cost Carriers',
             labels={'Average Fare': 'Average Fare ($)'},
             color='Carrier Type',
             color_discrete_map={'Large Carrier': '#EF553B', 'Low-Cost Carrier': '#00CC96'},
             text='Average Fare')

fig.update_traces(texttemplate='$%{text:.2f}', textposition='outside')
fig.update_layout(showlegend=False)
fig.show()

savings = df['fare_lg'].mean() - df['fare_low'].mean()
savings_pct = (savings / df['fare_lg'].mean()) * 100
print(f"\n💰 Average savings with low-cost carriers: ${savings:.2f} ({savings_pct:.1f}%)")

## 6. Price Range by Popular Routes

See the price variation on busy routes:

In [ ]:
# Find top 15 routes by passenger volume
top_routes = route_fares.nlargest(15, 'passengers')['route'].tolist()
df_top_routes = df[df['route'].isin(top_routes)]

fig = px.box(df_top_routes, 
             x='route', 
             y='fare',
             title='Price Range on Top 15 Busiest Routes',
             labels={'route': 'Route', 'fare': 'Fare ($)'},
             color='route')

fig.update_layout(showlegend=False, xaxis_tickangle=-45, height=600)
fig.update_xaxes(tickfont=dict(size=10))
fig.show()

## 7. Best Value Routes (Price per Mile)

Which routes give you the most distance for your money?

In [ ]:
# Calculate average price per mile by route
route_value = df.groupby('route').agg({
    'price_per_mile': 'mean',
    'nsmiles': 'mean',
    'fare': 'mean'
}).reset_index()

# Best value (lowest price per mile)
best_value = route_value.nsmallest(20, 'price_per_mile')

fig = px.bar(best_value, 
             x='price_per_mile', 
             y='route',
             orientation='h',
             title='Top 20 Best Value Routes (Lowest Price per Mile)',
             labels={'price_per_mile': 'Price per Mile ($)', 'route': 'Route'},
             color='price_per_mile',
             color_continuous_scale='Greens_r',
             hover_data={'fare': ':.2f', 'nsmiles': ':.0f'})

fig.update_layout(height=600, yaxis={'categoryorder':'total descending'})
fig.show()

## 💡 Key Insights Summary

In [ ]:
print("="*60)
print("📊 PRICE COMPARISON INSIGHTS")
print("="*60)

# Overall stats
print(f"\n💵 Overall Pricing:")
print(f"   • Average fare: ${df['fare'].mean():.2f}")
print(f"   • Median fare: ${df['fare'].median():.2f}")
print(f"   • Range: ${df['fare'].min():.2f} - ${df['fare'].max():.2f}")

# Carrier comparison
print(f"\n✈️ Carrier Comparison:")
print(f"   • Large carriers avg: ${df['fare_lg'].mean():.2f}")
print(f"   • Low-cost carriers avg: ${df['fare_low'].mean():.2f}")
print(f"   • Potential savings: ${df['fare_lg'].mean() - df['fare_low'].mean():.2f}")

# Most/least expensive
most_exp = route_fares.nlargest(1, 'fare').iloc[0]
least_exp = route_fares.nsmallest(1, 'fare').iloc[0]
print(f"\n🏆 Route Extremes:")
print(f"   • Most expensive: {most_exp['route']} (${most_exp['fare']:.2f})")
print(f"   • Cheapest: {least_exp['route']} (${least_exp['fare']:.2f})")

# Best value
best = route_value.nsmallest(1, 'price_per_mile').iloc[0]
print(f"\n💎 Best Value:")
print(f"   • Route: {best['route']}")
print(f"   • Price per mile: ${best['price_per_mile']:.3f}")

print("\n" + "="*60)